# 03 · Train the layout detector (title block, BOM table, notes, views)

YOLO (Ultralytics) detector for drawing regions. Two data sources:
1. **Synthetic** — rendered from the repo's generator, labels come free from the layout (no annotation cost).
2. **Real** — public Roboflow Universe engineering-drawing sets (title block / dimension / tolerance classes; free
   account + API key). Mixing real scans fixes the synthetic-only domain gap.

**Runtime:** GPU (T4 is enough). Export `detector.pt` into `backend/models/` — the backend switches to it automatically.

In [ ]:
# Get the repo, regenerate the synthetic corpus (deterministic, ~5 s) and fetch the public datasets (checksummed)
!git clone -q https://github.com/<you>/aeronexus-ai.git /content/aeronexus-ai || true
%cd /content/aeronexus-ai/backend
!pip -q install -r requirements.txt
!python scripts/generate_data.py --train 4000
!python scripts/fetch_public_data.py

In [ ]:
!pip -q install ultralytics pillow roboflow

In [ ]:
# Render synthetic sheets to PNG + YOLO labels from the heuristic layout (which is exact on synthetic sheets)
import json, os, random, sys
from PIL import Image, ImageDraw, ImageFont
sys.path.insert(0, ".")
from aeronexus.extraction.document import Sheet, HeuristicLayoutDetector
CLASSES = ["title_block", "bom_table", "notes", "view"]
det = HeuristicLayoutDetector()
root = "/content/yolo";
for sp in ["train", "val"]:
    os.makedirs(f"{root}/images/{sp}", exist_ok=True); os.makedirs(f"{root}/labels/{sp}", exist_ok=True)
rows = [json.loads(l) for l in open("data/train/drawings.jsonl")][:1500]
for i, d in enumerate(rows):
    s = Sheet.from_dict(d); W, H = 1280, int(1280 * s.height / s.width); k = W / s.width
    img = Image.new("L", (W, H), 255); dr = ImageDraw.Draw(img)
    dr.rectangle([8*k, 8*k, (s.width-8)*k, (s.height-8)*k], outline=0, width=2)
    for t in s.tokens:
        dr.text((t.bbox[0]*k, t.bbox[1]*k), t.text, fill=random.randint(0, 60))
    dr.rectangle([110*k, 190*k, 450*k, 390*k], outline=0, width=2)  # part outline
    sp = "val" if i % 10 == 0 else "train"
    img.save(f"{root}/images/{sp}/{s.id}.png")
    with open(f"{root}/labels/{sp}/{s.id}.txt", "w") as f:
        for r in det(s):
            x, y, w, h = r.bbox; pad = 8
            cx, cy = (x - pad + (w + 2*pad)/2)/s.width, (y - pad + (h + 2*pad)/2)/s.height
            f.write(f"{CLASSES.index(r.label)} {cx:.5f} {cy:.5f} {(w+2*pad)/s.width:.5f} {(h+2*pad)/s.height:.5f}\n")
open(f"{root}/data.yaml", "w").write(f"path: {root}\ntrain: images/train\nval: images/val\nnames: {CLASSES}\n")
print("rendered", len(rows))

In [ ]:
# (optional) add a real scanned-drawing set from Roboflow Universe (needs a free API key)
import os
if os.environ.get("ROBOFLOW_API_KEY"):
    from roboflow import Roboflow
    rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
    # e.g. workspace "trial-yolo", project "eng_drawing-idyau" — check the dataset's license on its page first
    real = rf.workspace("trial-yolo").project("eng_drawing-idyau").version(1).download("yolov8")
    print("downloaded to", real.location, "— merge its title_block class into data.yaml or fine-tune in a 2nd stage")

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
res = model.train(data="/content/yolo/data.yaml", epochs=40, imgsz=1280, batch=16, seed=0, patience=10)
metrics = model.val(); print(metrics.box.map50, metrics.box.map)

In [ ]:
import shutil
best = model.trainer.best; shutil.copy(best, "/content/detector.pt")
ARTIFACTS = ["/content/detector.pt"]

In [ ]:
# Save artifacts to Google Drive (survives the Colab session; avoids flaky browser downloads of big files).
# In VS Code: python scripts/import_models.py <the downloaded aeronexus_export.zip or Drive folder>
import hashlib, json, os, platform, shutil, sys, time
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/aeronexus/export"
os.makedirs(OUT, exist_ok=True)
for item in ARTIFACTS:                                   # files or folders produced above
    dst = os.path.join(OUT, os.path.basename(item.rstrip("/")))
    (shutil.copytree(item, dst, dirs_exist_ok=True) if os.path.isdir(item) else shutil.copy2(item, dst))
def sha(p):
    h = hashlib.sha256(); h.update(open(p, "rb").read()); return h.hexdigest()
import sklearn, numpy
man_path = os.path.join(OUT, "manifest.json")
man = json.load(open(man_path)) if os.path.exists(man_path) else {"artifacts": {}}
for root, _, fs in os.walk(OUT):
    for f in fs:
        if f != "manifest.json":
            p = os.path.join(root, f); man["artifacts"][os.path.relpath(p, OUT)] = sha(p)
man.update({"exported_at": time.strftime("%Y-%m-%dT%H:%M:%S"), "python": platform.python_version(),
            "versions": {"scikit-learn": sklearn.__version__, "numpy": numpy.__version__}})
json.dump(man, open(man_path, "w"), indent=1)
shutil.make_archive("/content/drive/MyDrive/aeronexus/aeronexus_export", "zip", OUT)
print("saved to Drive: MyDrive/aeronexus/aeronexus_export.zip  →", sorted(man["artifacts"])[:12])